# Z2005 — Week 14: Parallel and Randomized Algorithms

A self-study notebook covering Amdahl's Law, real parallel speedup, Las Vegas vs. Monte Carlo randomized algorithms, the parallel reduction pattern, and randomized selection (quickselect).

## Learning Objectives

By the end of this notebook you will be able to:

- State Amdahl's Law and use it to compute the theoretical speedup ceiling for a partially-parallel workload.
- Measure *real* parallel speedup on your own machine with `concurrent.futures`, and explain why measured speedup is usually worse than Amdahl's Law predicts.
- Distinguish **Las Vegas** algorithms (always correct, randomized running time) from **Monte Carlo** algorithms (randomized correctness, bounded running time), and classify a new algorithm as one or the other.
- Implement and verify a Monte Carlo primality test (Miller-Rabin) and explain how repeating trials amplifies confidence.
- Apply the parallel **reduction pattern** to sum, count, and max, and explain why naive averaging breaks under the same pattern.
- Implement randomized quickselect and explain why its expected running time is O(n), not O(n log n).

## How to use this notebook

Run the cells top to bottom. Markdown cells explain a concept; the code cell right after it is a fully worked example — read the comments, they explain *why* each line is there, not just *what* it does.

In the **Exercises** section, code cells contain a function signature, a docstring, and `# TODO: implement this` followed by `raise NotImplementedError`. Replace the `raise` with your implementation. The **Self-Check** cell right after each exercise uses `assert` statements: it raises an `AssertionError` (and tells you what's wrong) if your implementation is incorrect, and prints a friendly `✅` message if it's correct. Some self-checks for the randomized exercises run many trials and check a statistical property within a tolerance, rather than checking one exact answer — this is because randomized code will legitimately give different specific outputs on different runs.

The **Solutions** section at the very end has fully worked solutions to every exercise. Try the exercises yourself first — you learn more from a wrong attempt you had to debug than from reading someone else's working code.

## 1. Amdahl's Law: the theoretical ceiling on parallel speedup

Not every problem gets faster just because you throw more workers at it. Some part of almost every real workload is fundamentally sequential — setup, I/O, merging results at the end — and that part cannot be sped up no matter how many CPU cores you have.

**Amdahl's Law** formalizes this. If a fraction `p` of a program's running time can be parallelized (and the remaining `1 - p` cannot), then running it on `n` workers gives a speedup of

```
speedup(n) = 1 / ((1 - p) + p / n)
```

As `n → ∞`, the `p / n` term vanishes and the speedup approaches a hard ceiling of `1 / (1 - p)`. This is the single most important fact about parallel computing for a working engineer to internalize: if your workload is only 70% parallelizable, no amount of hardware will ever get you more than a 3.33x speedup, even with a million cores. Before buying more machines, ask what fraction of the work is actually the sequential 30% — that's usually the better investment.

**Common pitfall:** people intuitively expect speedup to scale linearly with the number of workers. It never does once you account for any sequential portion at all, and it also never does in practice because of scheduling and communication overhead (Section 2).

In [ ]:
def amdahl_speedup(p: float, n_workers: int) -> float:
    """Speedup from parallelizing a fraction `p` of the work across `n_workers`.

    p=0.0 means nothing is parallelizable (speedup always 1.0, regardless of n_workers).
    p=1.0 means everything is parallelizable (speedup == n_workers).
    """
    sequential_fraction = 1 - p
    return 1 / (sequential_fraction + p / n_workers)


def amdahl_ceiling(p: float) -> float:
    """The speedup limit as n_workers -> infinity: 1 / (1 - p)."""
    return 1 / (1 - p)


# Worked example: a workload where 70% of the time is parallelizable (p = 0.7),
# e.g. 30% of the time is spent on unavoidable sequential setup/merge work.
p = 0.7

# With exactly 1 worker there's no parallelism to exploit, so speedup must be 1.0.
assert round(amdahl_speedup(p, 1), 4) == 1.0

# The ceiling formula must agree with amdahl_speedup as n_workers grows very large.
assert round(amdahl_ceiling(p), 4) == round(1 / 0.3, 4)

print("Ceiling speedup at p=0.7:", round(amdahl_ceiling(p), 3))
print("Speedup at   4 workers:  ", round(amdahl_speedup(p, 4), 3))
print("Speedup at  16 workers:  ", round(amdahl_speedup(p, 16), 3))
print("Speedup at 100 workers:  ", round(amdahl_speedup(p, 100), 3))
# Notice how little is gained going from 16 to 100 workers -- we are already
# close to the 3.333x ceiling and throwing more cores at it barely helps.

## 2. A real, measured parallel speedup (and why it's usually disappointing)

Amdahl's Law describes a theoretical ceiling. In practice, real speedup is *worse* than the formula predicts, because the formula ignores the overhead of actually coordinating parallel work: starting worker processes, pickling arguments to send to them, and collecting results back. Python's `concurrent.futures.ProcessPoolExecutor` gives you real operating-system processes (so CPU-bound code genuinely runs concurrently, unlike threads under the GIL), but every one of those processes has real startup and communication cost.

**Important honesty note:** on a small workload, or on a shared/virtualized environment like Colab (which is often given only 1-2 CPU cores), the overhead of spinning up worker processes can easily *exceed* the time saved — you may see the "parallel" version run slower than the sequential one. That is not a bug in the demo; it is the single most important practical lesson about parallelism: **parallelizing a task is not free, and it is only worth it once the per-task work is large enough to amortize the coordination overhead.** The code below actually times both versions with `time.perf_counter()` on your machine right now — we do not print made-up numbers, we measure and report whatever your environment gives us.

**Common pitfall:** forgetting that `ProcessPoolExecutor` requires the worker function to be defined at module level (importable/picklable) — an inline `lambda` or a function defined inside another function will fail to pickle.

In [ ]:
import time
from concurrent.futures import ProcessPoolExecutor


def cpu_bound_chunk_work(chunk):
    """A deliberately CPU-heavy, embarrassingly-parallel task: sum of squares.

    Must be defined at module/top level (not nested) so it can be pickled
    and sent to worker processes.
    """
    return sum(x * x for x in chunk)


def run_sequential(chunks):
    return [cpu_bound_chunk_work(c) for c in chunks]


def run_parallel(chunks, max_workers=4):
    with ProcessPoolExecutor(max_workers=max_workers) as pool:
        # pool.map keeps result order matching input order, same as a
        # sequential list comprehension would -- important for correctness.
        return list(pool.map(cpu_bound_chunk_work, chunks))


# A workload large enough that the per-chunk work is not trivial.
data = list(range(4_000_000))
n_chunks = 4
chunks = [data[i::n_chunks] for i in range(n_chunks)]

start = time.perf_counter()
sequential_result = run_sequential(chunks)
sequential_time = time.perf_counter() - start

try:
    start = time.perf_counter()
    parallel_result = run_parallel(chunks, max_workers=n_chunks)
    parallel_time = time.perf_counter() - start
    assert sequential_result == parallel_result  # same answer, different execution strategy
    print(f"Sequential time: {sequential_time:.4f}s")
    print(f"Parallel time:   {parallel_time:.4f}s")
    if parallel_time < sequential_time:
        print(f"Measured speedup: {sequential_time / parallel_time:.2f}x")
    else:
        print("No speedup measured here -- process-pool overhead exceeded the time saved.")
        print("This is expected on small workloads or environments with few CPU cores (e.g. Colab).")
except (OSError, RuntimeError) as exc:
    # Some sandboxed/restricted notebook environments do not permit spawning
    # subprocesses at all. Report this honestly instead of failing silently.
    print("Could not start a process pool in this environment:", exc)
    print("Sequential result is still valid:", sequential_result[:2], "...")

## 3. Las Vegas vs. Monte Carlo algorithms

Randomized algorithms fall into two families, and confusing them is a genuinely common and serious mistake:

- **Las Vegas algorithms** are *always correct*; the randomness affects only their *running time*. Randomized quicksort is the canonical example: no matter which pivots the random number generator chooses, the output is always a correctly sorted array — the randomness just protects the *expected* running time from an adversarial worst-case input (e.g. an already-sorted array, which would make a fixed first-element pivot choice degrade to O(n²)).
- **Monte Carlo algorithms** always run in bounded time, but *may return an incorrect answer* with some (typically small, controllable) probability. Miller-Rabin primality testing (Section 4) is the canonical example: it always finishes fast, but a composite number can — rarely — pass all the checks and be misreported as "probably prime."

**Why this distinction matters:** you would never accept "this sort might return an unsorted array" — correctness must be guaranteed for sorting, so if you need randomness there, it must be Las Vegas. But you might well accept "there's a 1 in 2^40 chance this claimed-prime number is actually composite" for something like generating an RSA key, because that failure probability is astronomically smaller than the chance of, say, a hardware bit-flip error — that's what makes Monte Carlo acceptable for primality testing.

**Common pitfall:** claiming an algorithm is "randomized, so it might be wrong sometimes" when it is actually Las Vegas (always correct) — or the reverse, assuming a Monte Carlo algorithm's output is trustworthy without checking its error probability.

In [ ]:
import random


def randomized_quicksort(arr):
    """Las Vegas sort: ALWAYS returns a correctly sorted list.

    The random pivot choice only affects how long it takes, never whether
    the output is correct -- this is what makes it Las Vegas, not Monte Carlo.
    """
    if len(arr) <= 1:
        return arr
    pivot = random.choice(arr)                 # <- the only randomized step
    less = [x for x in arr if x < pivot]
    equal = [x for x in arr if x == pivot]
    greater = [x for x in arr if x > pivot]
    return randomized_quicksort(less) + equal + randomized_quicksort(greater)


# No matter how many times we run this, the OUTPUT is always correctly sorted,
# even though the recursion tree shape (and hence the running time) differs
# from run to run because of the random pivot choices.
test_cases = [
    [],
    [1],
    [5, 3, 1, 4, 2],
    list(range(20, 0, -1)),      # already-sorted-descending: adversarial for a fixed-pivot quicksort
    [7, 7, 7, 3, 3, 9, 1],       # duplicates
]
for case in test_cases:
    for _trial in range(20):     # run many times: correctness must hold on every single run
        assert randomized_quicksort(case) == sorted(case)

print("randomized_quicksort produced a correctly sorted list on every one of 20 trials per test case")
print("This is the Las Vegas guarantee: randomness affects speed, never correctness.")

## 4. Monte Carlo primality testing: Miller-Rabin

Miller-Rabin tests whether `n` is prime by writing `n - 1 = 2^r * d` (with `d` odd) and repeatedly checking whether a randomly-chosen "witness" `a` *proves* `n` is composite. If a witness proves compositeness, `n` is definitely composite — no doubt. If no witness among `k` random trials proves compositeness, `n` is reported as "probably prime," and the probability that a composite number wrongly passes all `k` trials is at most `4^(-k)` — with `k = 20` (the default below) that's about 1 in a trillion, which is why this test is used to generate real-world cryptographic primes.

This is what makes Miller-Rabin Monte Carlo rather than Las Vegas: it always finishes in bounded time (no retry loops that could run forever), but "probably prime" is genuinely a probabilistic claim, not a proof — this is the **amplification** idea: each additional independent trial multiplies the error probability down, so you choose `k` based on how much error you're willing to tolerate.

**Common pitfall:** treating "probably prime" as a mathematical proof rather than a high-confidence statistical claim — for anything security-critical you should be explicit about `k` and the resulting error bound, not just call `is_probably_prime` once and trust it blindly.

In [ ]:
def witness_proves_composite(n, a, d, r):
    """Return True if witness `a` proves n is composite (via n-1 = 2^r * d)."""
    x = pow(a, d, n)                 # a^d mod n, computed efficiently
    if x == 1 or x == n - 1:
        return False                 # inconclusive -- a does not prove compositeness
    for _ in range(r - 1):
        x = pow(x, 2, n)             # repeated squaring, mod n
        if x == n - 1:
            return False             # inconclusive
    return True                      # every check failed -- a PROVES n is composite


def is_probably_prime(n, k=20):
    """Monte Carlo primality test. False negatives never happen (composite
    numbers are never reported prime with certainty violated -- see error
    bound above); false positives happen with probability at most 4^-k."""
    if n < 2:
        return False
    for small_prime in [2, 3, 5, 7, 11, 13]:
        if n % small_prime == 0:
            return n == small_prime  # n IS one of these primes, or a multiple of one (composite)
    d, r = n - 1, 0
    while d % 2 == 0:                # factor out powers of 2: n - 1 = 2^r * d, d odd
        d //= 2
        r += 1
    for _ in range(k):                          # k independent random trials
        a = random.randrange(2, n - 1)           # random witness candidate
        if witness_proves_composite(n, a, d, r):
            return False              # one witness proving compositeness is enough to be certain
    return True                       # survived all k trials: "probably prime"


# Worked trace: n = 221 = 13 x 17, so n - 1 = 220 = 2^2 * 55 -> d=55, r=2
assert witness_proves_composite(221, 174, 55, 2) == False   # a=174 does NOT prove composite
assert witness_proves_composite(221, 137, 55, 2) == True    # a=137 PROVES composite

# Second worked trace: n = 561, a Carmichael number (fools the weaker Fermat test,
# but not Miller-Rabin) -- n - 1 = 560 = 2^4 * 35 -> d=35, r=4
assert witness_proves_composite(561, 2, 35, 4) == True

for prime in [17, 97, 7919]:
    assert is_probably_prime(prime)
for composite in [221, 1000, 8051, 561]:
    assert not is_probably_prime(composite)

print("Miller-Rabin checks passed on both worked traces and several known primes/composites")

## 5. The parallel reduction pattern

Many parallel computations follow the same shape: split the data into independent chunks, compute a *partial* result on each chunk independently (this is the part that parallelizes), then *combine* the partial results into a final answer (this final combine step is typically sequential, and is exactly the `(1 - p)` term from Amdahl's Law).

Sum, count, and max all reduce cleanly this way: `sum(chunks) == sum(sum(chunk) for chunk in chunks)`, and similarly for count and max. Merge sort also fits this pattern — sort each half independently (parallelizable), then merge the two sorted halves (sequential, and this merge step is exactly why merge sort's parallel speedup falls short of what dividing the *sorting* work alone would suggest).

**Common pitfall — the averaging trap:** you *cannot* average partial averages and get the right overall average unless every chunk has the same size (`average([10,20,30]) == 20`, `average([5,15]) == 10`, but the true average of all five numbers is `16`, not `(20+10)/2 == 15`). The fix is to reduce `(sum, count)` pairs instead of averages directly, and divide only once at the very end — averaging is not naturally reducible; sum and count are.

In [ ]:
def merge(a, b):
    """Merge two already-sorted lists into one sorted list."""
    result, i, j = [], 0, 0
    while i < len(a) and j < len(b):
        if a[i] <= b[j]:
            result.append(a[i]); i += 1
        else:
            result.append(b[j]); j += 1
    result.extend(a[i:])   # append whatever is left of `a`
    result.extend(b[j:])   # append whatever is left of `b`
    return result


def merge_sort(arr):
    """Sequential merge sort. The two recursive calls are independent of
    each other -- that independence is exactly what makes them parallelizable
    -- but the final `merge` call is inherently sequential."""
    if len(arr) <= 1:
        return arr
    mid = len(arr) // 2
    left = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])
    return merge(left, right)   # <- the sequential "combine" step


arr = [38, 27, 43, 3, 9, 82, 10]
assert merge_sort(arr) == sorted(arr)

# --- Reduction pattern: parallel sum ---
def partial_sum(chunk):
    return sum(chunk)

data = list(range(1, 100_001))
chunks = [data[i::4] for i in range(4)]      # 4 independent chunks
partials = [partial_sum(c) for c in chunks]  # the parallelizable step
parallel_total = sum(partials)               # the sequential combine step
assert parallel_total == sum(data) == 5_000_050_000

# --- The averaging pitfall: WRONG way (averaging partial averages) ---
avg_chunks = [[10, 20, 30], [5, 15]]
wrong_average = sum(sum(c) / len(c) for c in avg_chunks) / len(avg_chunks)
true_average = sum(x for c in avg_chunks for x in c) / sum(len(c) for c in avg_chunks)
assert wrong_average != true_average   # demonstrates the trap is real, not hypothetical

# --- The averaging pitfall: RIGHT way (reduce (sum, count) pairs, divide once at the end) ---
def chunk_sum_and_count(chunk):
    return sum(chunk), len(chunk)

def parallel_average(chunks):
    partials = [chunk_sum_and_count(c) for c in chunks]
    total = sum(s for s, _ in partials)
    count = sum(c for _, c in partials)
    return total / count

assert parallel_average(avg_chunks) == true_average == 16.0
print("Merge sort, parallel-sum, and the averaging-pitfall demo all check out")
print(f"Naive (wrong) average of averages: {wrong_average}, correct average: {true_average}")

## 6. Randomized selection: quickselect

Quickselect answers "what is the k-th smallest element?" without fully sorting the array. It reuses quicksort's partition idea: pick a random pivot, partition the array into elements less than, equal to, and greater than the pivot, then recurse into *only the one partition that contains the k-th position* — unlike quicksort, which must recurse into both halves.

Because it only ever recurses into one side, quickselect's expected running time is **O(n)**, not **O(n log n)** — each partition step does O(n) work, but the total work across all levels of recursion is a geometric series (roughly n + n/2 + n/4 + ... ) that sums to O(n) rather than growing with the number of levels. Finding a single value (like the median) is therefore asymptotically cheaper than fully sorting the array just to index into it, even though `sorted(arr)[k]` is simpler to write and is often fast enough in practice for small `n`.

**Common pitfall:** quickselect's O(n) bound is an *expected* (average-case) bound, exactly like randomized quicksort's O(n log n) — an adversary who knows your pivot strategy in advance can still force worst-case O(n²) behavior on a *fixed* (non-random) pivot rule; choosing the pivot randomly is what defeats that adversary, the same idea as Section 3.

In [ ]:
def quickselect(arr, k):
    """Return the k-th smallest element of arr (0-indexed): quickselect(arr, 0)
    is the minimum, quickselect(arr, len(arr)-1) is the maximum."""
    if len(arr) == 1:
        return arr[0]
    pivot = random.choice(arr)                     # random pivot defeats adversarial inputs
    lows = [x for x in arr if x < pivot]
    highs = [x for x in arr if x > pivot]
    pivots = [x for x in arr if x == pivot]
    if k < len(lows):
        return quickselect(lows, k)                 # k-th element is in the "less than" partition
    elif k < len(lows) + len(pivots):
        return pivot                                 # k-th element IS the pivot (handles duplicates)
    else:
        # k-th element is in the "greater than" partition; shift k to
        # account for the elements we have already ruled out.
        return quickselect(highs, k - len(lows) - len(pivots))


def quickselect_median(arr):
    return quickselect(arr, len(arr) // 2)


qs_arr = [7, 10, 4, 3, 20, 15]
qs_sorted = sorted(qs_arr)
for k in range(len(qs_arr)):
    assert quickselect(qs_arr, k) == qs_sorted[k]   # check EVERY rank, not just one

median_arr = [9, 1, 8, 2, 7, 3, 6]
assert quickselect_median(median_arr) == sorted(median_arr)[len(median_arr) // 2] == 6

for test in [[5, 3, 1, 4, 2], [9, 9, 1], [100], [7, 2, 9, 4, 6], [3, 3, 3, 3, 3]]:
    assert quickselect_median(test) == sorted(test)[len(test) // 2]

print("Quickselect checks passed for every rank and several duplicate-heavy inputs")

## 7. Rabin-Karp string matching with rolling hashes

Naively checking whether a pattern of length `m` occurs in a text of length `n` by sliding the pattern across every position and comparing character-by-character costs `O(nm)` in the worst case (e.g. text `"aaaa...a"`, pattern `"aaa...ab"`). Rabin-Karp speeds this up with a **hash-then-verify** strategy: instead of comparing strings directly at every position, compare cheap numeric hashes first, and only fall back to a full character-by-character comparison when the hashes match.

The trick that makes this fast is a **rolling hash**: the hash of the window starting at position `i + 1` can be computed from the hash of the window starting at position `i` in `O(1)` time, rather than recomputing it from scratch. Treating the `m` characters of a window as digits of a base-`B` number, sliding the window by one position means "removing" the leading digit and "appending" a new trailing digit — both cheap arithmetic operations (all done modulo a large prime to keep the numbers bounded).

Because hashes are computed modulo some `mod`, two *different* strings can occasionally hash to the same value — a **spurious hit** (hash collision). Rabin-Karp handles this by always verifying a hash match with a real character-by-character comparison before reporting a match; this is why the algorithm is always correct, never just probably correct.

This gives Rabin-Karp an **expected running time of `O(n + m)`**: computing all the rolling hashes takes `O(n)`, and as long as spurious hits are rare, the verification step adds only `O(m)` total. But the **worst case is still `O(nm)`** — a pathological input (or an unlucky choice of `mod`) could make *every* window hash-match the pattern, forcing a full `O(m)` verification at each of the `O(n)` positions. In practice, a large prime modulus makes this vanishingly unlikely for real-world inputs.

In [ ]:
def rabin_karp_search(text: str, pattern: str, base: int = 256, mod: int = 1_000_000_007) -> list:
    """Return a list of all starting indices where pattern occurs in text.

    Uses a rolling hash to compare candidate windows to the pattern in O(1)
    amortized time per window, falling back to a full character comparison
    only when the hashes match (hash-then-verify).

    >>> rabin_karp_search("abracadabra", "abra")
    [0, 7]
    """
    n, m = len(text), len(pattern)
    if m == 0 or m > n:
        return []

    # high_order = base^(m-1) mod mod -- the "place value" of the leading
    # character in an m-character window, needed to remove it when rolling.
    high_order = pow(base, m - 1, mod)

    pattern_hash = 0
    window_hash = 0
    for i in range(m):
        pattern_hash = (pattern_hash * base + ord(pattern[i])) % mod
        window_hash = (window_hash * base + ord(text[i])) % mod

    matches = []
    for i in range(n - m + 1):
        # Hash-then-verify: only pay for a full string comparison when the
        # cheap hash comparison already matched (guards against spurious
        # hits / hash collisions -- this verification is what keeps the
        # algorithm always correct despite using a probabilistic shortcut).
        if window_hash == pattern_hash and text[i:i + m] == pattern:
            matches.append(i)
        if i < n - m:
            # Roll the hash forward by one position: drop the leading
            # character's contribution, shift, then add the new trailing
            # character -- all O(1), which is what gives Rabin-Karp its
            # expected O(n + m) running time.
            window_hash = (window_hash - ord(text[i]) * high_order) % mod
            window_hash = (window_hash * base + ord(text[i + m])) % mod
            window_hash %= mod
    return matches


# Worked example: find every occurrence of "abra" in a short text.
demo_text = "abracadabra abracadabra"
demo_pattern = "abra"
print(f"text:    {demo_text!r}")
print(f"pattern: {demo_pattern!r}")
print("match starting indices:", rabin_karp_search(demo_text, demo_pattern))

## 8. Backtracking: the N-Queens problem

Backtracking is a general strategy for problems that ask "find all (or one) arrangement of choices that satisfies a set of constraints." The idea is **incremental candidate construction**: build a solution one decision at a time, and after each decision, check whether the *partial* solution still satisfies the constraints so far. If a partial solution can never be extended into a full valid solution, **prune** that branch immediately (don't bother exploring anything built on top of it) and **undo** the last decision (backtrack) to try the next option instead.

The N-Queens problem is the classic teaching example: place `N` queens on an `N x N` chessboard so that no two queens attack each other (no two share a row, a column, or a diagonal). Rather than generating all `N^N` ways to place `N` queens on `N^2` squares and checking each one, backtracking places queens one row at a time — one queen per row is forced by the "no shared row" constraint — and checks, before placing a queen in a column, whether that column or either diagonal is already attacked by a previously placed queen. If it is, that column is skipped entirely for this row: an invalid partial placement is pruned long before the remaining rows are ever considered, which is what makes backtracking dramatically faster than brute force in practice, even though its worst-case complexity is still exponential.

This is a general pattern you will see again and again: **choose** a candidate for the next decision, **check** whether it's still consistent with everything placed so far, **recurse** into the next decision if it is, and **undo** the choice (backtrack) after exploring it (whether or not it led to a solution) so the next candidate can be tried cleanly.

In [ ]:
def solve_n_queens(n: int) -> list:
    """Return a list of all solutions to the N-Queens problem for board size n.

    Each solution is a tuple `columns` of length n, where columns[row] is the
    column index of the queen placed in that row (one queen per row, since
    two queens can never share a row).

    >>> len(solve_n_queens(4))
    2
    """
    solutions = []
    columns = [-1] * n   # columns[row] = column of the queen in that row (so far)

    def is_safe(row: int, col: int) -> bool:
        """Would placing a queen at (row, col) conflict with any already-placed queen?"""
        for r in range(row):
            c = columns[r]
            if c == col or abs(c - col) == abs(r - row):   # same column or same diagonal
                return False
        return True

    def backtrack(row: int) -> None:
        if row == n:
            # Every row has a safely-placed queen -- record this full solution.
            solutions.append(tuple(columns))
            return
        for col in range(n):
            if is_safe(row, col):
                columns[row] = col          # choose
                backtrack(row + 1)          # recurse
                columns[row] = -1           # undo (backtrack) before trying the next column

    backtrack(0)
    return solutions


# Worked example: N=4 is small enough to print and verify by hand.
n = 4
solutions = solve_n_queens(n)
print(f"N-Queens solutions for N={n}: {len(solutions)} found")
for columns in solutions:
    print("  columns per row:", columns)

# N=8 (the classic case) still runs essentially instantly thanks to pruning.
solutions_8 = solve_n_queens(8)
print(f"N-Queens solutions for N=8: {len(solutions_8)} found (should be 92)")

## 9. The Traveling Salesperson Problem (TSP): brute force and a practical heuristic

The Traveling Salesperson Problem asks: given a set of cities and the distance between every pair, what is the shortest possible route that visits every city exactly once and returns to the starting city? TSP is **NP-hard** — no known algorithm solves it exactly in polynomial time in the general case, and the most direct exact approach, trying every possible visiting order, is `O(n!)` (there are `(n-1)!/2` distinct tours to check, since a tour and its reverse have the same length, and the starting city can be fixed without loss of generality). This grows explosively: 10 cities already means over 180,000 distinct tours, and 15 cities means over 43 billion — brute force is only practical for very small instances.

Because exact solutions are so expensive, real applications almost always use a **heuristic**: an algorithm that finds a *good* (not necessarily optimal) tour quickly. The **nearest-neighbor heuristic** is the simplest one: starting from some city, repeatedly travel to the closest city not yet visited, until every city has been visited, then return to the start. It runs in `O(n^2)` time (checking the nearest unvisited city from each of `n` positions costs `O(n)`), a huge improvement over `O(n!)`, but it offers no optimality guarantee — a locally "greedy" choice early in the tour can force an expensive detour near the end. In practice this trade-off is usually worthwhile: for many real distance layouts, nearest-neighbor tours land within a reasonable factor of optimal, and the difference in running time makes it the only option once the number of cities grows beyond what brute force can handle.

In [ ]:
import itertools
import math


def tsp_brute_force(distance_matrix: list) -> tuple:
    """Exact TSP solution by trying every possible visiting order.

    distance_matrix[i][j] is the distance from city i to city j.
    City 0 is fixed as the start/end city (a tour and its rotations are
    equivalent, so this loses no generality but avoids redundant work).

    Returns (best_tour, best_cost), where best_tour is a list of city
    indices starting and ending at city 0.

    O(n!) -- only practical for a handful of cities.
    """
    n = len(distance_matrix)
    other_cities = list(range(1, n))
    best_tour, best_cost = None, math.inf
    for perm in itertools.permutations(other_cities):
        tour = [0] + list(perm) + [0]
        cost = sum(distance_matrix[tour[i]][tour[i + 1]] for i in range(len(tour) - 1))
        if cost < best_cost:
            best_cost, best_tour = cost, tour
    return best_tour, best_cost


def tsp_nearest_neighbor(distance_matrix: list, start: int = 0) -> tuple:
    """Fast TSP heuristic: always travel to the nearest unvisited city.

    Returns (tour, cost). Not guaranteed optimal, but runs in O(n^2) instead
    of O(n!), making it usable on instances far too large for brute force.
    """
    n = len(distance_matrix)
    unvisited = set(range(n)) - {start}
    tour = [start]
    current = start
    total_cost = 0.0
    while unvisited:
        nearest = min(unvisited, key=lambda city: distance_matrix[current][city])
        total_cost += distance_matrix[current][nearest]
        tour.append(nearest)
        unvisited.remove(nearest)
        current = nearest
    total_cost += distance_matrix[current][start]   # return to the start city
    tour.append(start)
    return tour, total_cost


# Worked example: 5 cities with a symmetric distance matrix.
distances = [
    [0, 10, 15, 20, 25],
    [10, 0, 35, 25, 30],
    [15, 35, 0, 30, 20],
    [20, 25, 30, 0, 15],
    [25, 30, 20, 15, 0],
]

exact_tour, exact_cost = tsp_brute_force(distances)
print(f"Brute force (exact): tour={exact_tour}, cost={exact_cost}")

heuristic_tour, heuristic_cost = tsp_nearest_neighbor(distances)
print(f"Nearest-neighbor (heuristic): tour={heuristic_tour}, cost={heuristic_cost}")

print(f"Heuristic is within {heuristic_cost / exact_cost:.2f}x of optimal on this instance")

## Exercises

### Exercise 1 — Amdahl's Law: minimum parallel fraction

Write `min_parallel_fraction_for_speedup(target_speedup)` that returns the smallest fraction `p` (parallelizable fraction, `0 <= p < 1`) such that the *ceiling* speedup `1 / (1 - p)` is at least `target_speedup`, rounded to 4 decimal places.

Example: `min_parallel_fraction_for_speedup(2.0)` should return `0.5` (because `1 / (1 - 0.5) == 2.0` exactly).

In [ ]:
def min_parallel_fraction_for_speedup(target_speedup: float) -> float:
    """Smallest p in [0, 1) such that 1 / (1 - p) >= target_speedup, rounded to 4 dp.

    >>> min_parallel_fraction_for_speedup(2.0)
    0.5
    """
    # TODO: implement this
    raise NotImplementedError

#### Self-Check 1

In [ ]:
assert min_parallel_fraction_for_speedup(2.0) == 0.5
assert min_parallel_fraction_for_speedup(4.0) == 0.75
assert min_parallel_fraction_for_speedup(1.0) == 0.0
assert min_parallel_fraction_for_speedup(10.0) == 0.9
print("✅ Exercise 1 passed")

### Exercise 2 — Parallel reduction: max and its index

Write `parallel_argmax(chunks)` that takes a list of chunks (each chunk a list of numbers) and returns a tuple `(max_value, chunk_index)` — the maximum value across all chunks, and *which chunk it came from* (0-indexed). This mirrors the reduction pattern from Section 5, but the combine step is slightly trickier than plain `max` because you must track provenance.

Example: `parallel_argmax([[3, 7], [10, 2], [5]])` should return `(10, 1)` because `10` is the overall maximum and it lives in chunk index `1`.

In [ ]:
def parallel_argmax(chunks: list) -> tuple:
    """Return (max_value, index_of_chunk_containing_it).

    >>> parallel_argmax([[3, 7], [10, 2], [5]])
    (10, 1)
    """
    # TODO: implement this
    # Hint: compute the partial max of each chunk (the parallelizable step),
    # then combine the (partial_max, chunk_index) pairs (the sequential step).
    raise NotImplementedError

#### Self-Check 2

In [ ]:
assert parallel_argmax([[3, 7], [10, 2], [5]]) == (10, 1)
assert parallel_argmax([[1], [2], [3]]) == (3, 2)
assert parallel_argmax([[9, 9, 9], [1]]) == (9, 0)
assert parallel_argmax([[-5, -1], [-10, -2]]) == (-1, 0)
print("✅ Exercise 2 passed")

### Exercise 3 — Classify Las Vegas vs. Monte Carlo

Write `classify_algorithm(always_correct, bounded_time)` that takes two booleans describing a randomized algorithm's properties and returns the string `"Las Vegas"`, `"Monte Carlo"`, or `"invalid"` according to these rules:

- `"Las Vegas"`: always produces a correct answer (`always_correct=True`), but running time is not necessarily bounded (`bounded_time=False`).
- `"Monte Carlo"`: running time is always bounded (`bounded_time=True`), but the answer is not guaranteed correct (`always_correct=False`).
- `"invalid"`: any other combination (e.g. both correct AND always bounded describes a plain deterministic algorithm, not a meaningfully randomized one for this exercise's purposes; neither correct nor bounded is not a useful algorithm at all).

In [ ]:
def classify_algorithm(always_correct: bool, bounded_time: bool) -> str:
    """Classify a randomized algorithm as 'Las Vegas', 'Monte Carlo', or 'invalid'.

    >>> classify_algorithm(always_correct=True, bounded_time=False)
    'Las Vegas'
    >>> classify_algorithm(always_correct=False, bounded_time=True)
    'Monte Carlo'
    """
    # TODO: implement this
    raise NotImplementedError

#### Self-Check 3

In [ ]:
assert classify_algorithm(always_correct=True, bounded_time=False) == "Las Vegas"
assert classify_algorithm(always_correct=False, bounded_time=True) == "Monte Carlo"
assert classify_algorithm(always_correct=True, bounded_time=True) == "invalid"
assert classify_algorithm(always_correct=False, bounded_time=False) == "invalid"
# Sanity check against the algorithms studied above:
assert classify_algorithm(always_correct=True, bounded_time=False) == "Las Vegas"   # randomized quicksort
assert classify_algorithm(always_correct=False, bounded_time=True) == "Monte Carlo"  # Miller-Rabin
print("✅ Exercise 3 passed")

### Exercise 4 — Monte Carlo estimation of pi

Write `estimate_pi(n_samples)` that estimates the value of π using the classic Monte Carlo "darts on a square" method: generate `n_samples` random points `(x, y)` with `x, y` uniform in `[-1, 1]`, count what fraction land inside the unit circle (`x**2 + y**2 <= 1`), and return `4 * (fraction inside the circle)` — this converges to π as `n_samples` grows, because the circle's area is π and the square's area is 4.

This is your first Monte Carlo algorithm that estimates a *numeric value* rather than answering a yes/no question — same family as Miller-Rabin (bounded running time, no guarantee of exact correctness), but the "error" here is a continuous approximation error rather than a probability of a wrong boolean answer.

In [ ]:
def estimate_pi(n_samples: int) -> float:
    """Monte Carlo estimate of pi using random points in [-1, 1] x [-1, 1].

    Larger n_samples gives a more accurate (but still random) estimate.
    """
    # TODO: implement this
    # Hint: use random.uniform(-1, 1) for each coordinate, and count how many
    # of the n_samples points satisfy x**2 + y**2 <= 1.
    raise NotImplementedError

#### Self-Check 4

In [ ]:
# Statistical self-check: a single run of a Monte Carlo estimator is random,
# so we check that the estimate is within a generous tolerance of the true
# value, using a reasonably large sample size to keep variance manageable.
import math

estimate = estimate_pi(200_000)
assert abs(estimate - math.pi) < 0.05, f"estimate {estimate} too far from pi={math.pi}"

# A tiny sample should still land in a *much* wider plausible range (0 to 4,
# since 4*fraction can never exceed 4 or go below 0).
tiny_estimate = estimate_pi(10)
assert 0.0 <= tiny_estimate <= 4.0
print(f"✅ Exercise 4 passed (estimate with 200,000 samples: {estimate:.4f}, true pi: {math.pi:.4f})")

### Exercise 5 — Quickselect-based top-k (harder)

Write `top_k_smallest(arr, k)` that returns the `k` smallest elements of `arr`, **in sorted order**, using `quickselect` (already defined above — reuse it) to avoid a full `O(n log n)` sort of the *entire* array. Only the `k` smallest elements need to end up sorted; the approach should still be asymptotically cheaper than `sorted(arr)[:k]` for the *selection* step, even though the final small slice must be sorted.

In [ ]:
def top_k_smallest(arr: list, k: int) -> list:
    """Return the k smallest elements of arr, in ascending sorted order.

    >>> top_k_smallest([7, 10, 4, 3, 20, 15], 3)
    [3, 4, 7]

    Hint: quickselect(arr, k - 1) gives you the k-th smallest VALUE (the
    "boundary"), which you can use to partition arr into the k smallest
    elements versus the rest, then sort only that small slice.
    """
    # TODO: implement this
    raise NotImplementedError

#### Self-Check 5

In [ ]:
assert top_k_smallest([7, 10, 4, 3, 20, 15], 3) == [3, 4, 7]
assert top_k_smallest([5, 3, 1, 4, 2], 5) == [1, 2, 3, 4, 5]
assert top_k_smallest([9, 9, 1, 1, 5], 3) == [1, 1, 5]
assert top_k_smallest([100], 1) == [100]
assert top_k_smallest([7, 2, 9, 4, 6], 0) == []
print("✅ Exercise 5 passed")

### Exercise 6 — Rabin-Karp: count pattern occurrences

Write `count_pattern_occurrences(text, pattern)` that returns the *number* of times `pattern` occurs in `text` (including overlapping occurrences), using the rolling-hash technique from `rabin_karp_search` above. Do not use Python's built-in `str.count` or `in`/`find` -- reuse the rolling-hash approach.

Example: `count_pattern_occurrences("aaaa", "aa")` should return `3` (occurrences start at index 0, 1, and 2 -- they overlap).

In [ ]:
def count_pattern_occurrences(text: str, pattern: str) -> int:
    """Count occurrences of pattern in text (including overlaps), via rolling hash.

    >>> count_pattern_occurrences("aaaa", "aa")
    3
    """
    # TODO: implement this
    # Hint: reuse the rolling-hash idea from rabin_karp_search -- you can
    # call rabin_karp_search directly and return the length of the result.
    raise NotImplementedError

#### Self-Check 6

In [ ]:
assert count_pattern_occurrences("aaaa", "aa") == 3
assert count_pattern_occurrences("abracadabra", "abra") == 2
assert count_pattern_occurrences("abcabcabc", "abc") == 3
assert count_pattern_occurrences("xyz", "abc") == 0
assert count_pattern_occurrences("aaaa", "aaaa") == 1
print("✅ Exercise 6 passed")

### Exercise 7 — Count N-Queens solutions

Write `count_n_queens_solutions(n)` that returns the *number* of distinct solutions to the N-Queens problem for a board of size `n`, without necessarily storing every full solution (though reusing the backtracking structure from `solve_n_queens` and just returning `len(...)` is a perfectly valid implementation).

Example: `count_n_queens_solutions(4)` should return `2`.

In [ ]:
def count_n_queens_solutions(n: int) -> int:
    """Return the number of distinct N-Queens solutions for board size n.

    >>> count_n_queens_solutions(4)
    2
    """
    # TODO: implement this
    # Hint: this can reuse the exact same backtracking structure as
    # solve_n_queens -- you just need a count instead of a list of solutions.
    raise NotImplementedError

#### Self-Check 7

In [ ]:
assert count_n_queens_solutions(1) == 1
assert count_n_queens_solutions(2) == 0
assert count_n_queens_solutions(3) == 0
assert count_n_queens_solutions(4) == 2
assert count_n_queens_solutions(5) == 10
print("✅ Exercise 7 passed")

### Exercise 8 — Best nearest-neighbor starting city

The nearest-neighbor heuristic's tour quality can depend on which city you start from. Write `best_nearest_neighbor_tour(distance_matrix)` that runs `tsp_nearest_neighbor` starting from *every* city in turn, and returns the `(tour, cost)` pair with the lowest cost found across all starting cities.

This does not turn nearest-neighbor into an exact algorithm (it is still just a heuristic, and the best of several nearest-neighbor runs is not guaranteed to be the true optimum), but trying multiple starting points is a common, cheap way to improve a heuristic's typical result.

In [ ]:
def best_nearest_neighbor_tour(distance_matrix: list) -> tuple:
    """Run nearest-neighbor from every possible starting city; return the best.

    Returns (tour, cost) for whichever starting city produced the lowest cost.
    """
    # TODO: implement this
    # Hint: loop start in range(len(distance_matrix)), call
    # tsp_nearest_neighbor(distance_matrix, start=start) for each, and keep
    # track of the (tour, cost) pair with the smallest cost seen so far.
    raise NotImplementedError

#### Self-Check 8

In [ ]:
small_distances = [
    [0, 10, 15, 20, 25],
    [10, 0, 35, 25, 30],
    [15, 35, 0, 30, 20],
    [20, 25, 30, 0, 15],
    [25, 30, 20, 15, 0],
]
exact_tour, exact_cost = tsp_brute_force(small_distances)
best_tour, best_cost = best_nearest_neighbor_tour(small_distances)

# The best nearest-neighbor tour can never beat the true optimum, but on
# this small instance it should be at least as good as starting from city 0
# alone, and never worse than the exact optimum.
_, start0_cost = tsp_nearest_neighbor(small_distances, start=0)
assert best_cost <= start0_cost
assert best_cost >= exact_cost
assert best_tour[0] == best_tour[-1]          # tour returns to its start
assert len(set(best_tour[:-1])) == len(small_distances)   # visits every city once
print(f"✅ Exercise 8 passed (best nearest-neighbor cost: {best_cost}, optimal: {exact_cost})")

## Quiz

**Q1.** A workload is 40% parallelizable (`p = 0.4`). What is the maximum possible speedup, no matter how many workers you use?

<details><summary>Show answer</summary>1 / (1 - 0.4) = 1 / 0.6 ≈ 1.67x. Even with infinite workers, you can never do better than about 1.67x speedup, because 60% of the work is fundamentally sequential.</details>

**Q2.** You run a parallel version of a function on 8 workers and it takes *longer* than the sequential version. Is your parallel code necessarily buggy?

<details><summary>Show answer</summary>Not necessarily. If each unit of work is small, the overhead of starting worker processes and communicating results can exceed the time saved by parallelizing — this is a real, common outcome, not a bug, and is exactly what Section 2's honest measurement demonstrates. It becomes a problem only if it happens on workloads that are actually large enough that parallelism should help.</details>

**Q3.** Is randomized quicksort a Las Vegas or a Monte Carlo algorithm, and why?

<details><summary>Show answer</summary>Las Vegas. It always produces a correctly sorted output — the randomness (pivot choice) only affects how long it takes to run, never whether the answer is correct.</details>

**Q4.** Why can't you compute the overall average of a dataset by averaging the per-chunk averages (when chunks have different sizes)?

<details><summary>Show answer</summary>Because a simple average of averages implicitly weights every chunk equally regardless of how many elements it contains, which is wrong when chunk sizes differ. The correct reduction is to combine (sum, count) pairs from each chunk, add up the sums and the counts separately, and divide only once at the very end.</details>

**Q5.** Rabin-Karp's hashes for two different windows happen to be equal, but the windows contain different text. What does Rabin-Karp do, and why is this necessary for correctness?

<details><summary>Show answer</summary>It falls back to a full character-by-character comparison of that window against the pattern before reporting a match (hash-then-verify). This is necessary because a hash match is only a *candidate* -- two different strings can collide to the same hash value (a spurious hit) purely by chance, and without verifying, Rabin-Karp could report false matches.</details>

**Q6.** Why is Rabin-Karp's expected running time O(n + m) but its worst-case running time O(nm)?

<details><summary>Show answer</summary>Computing the rolling hash for every window is O(n) total, and normally very few (if any) windows have a spurious hash match, so verification adds only O(m) overall -- giving expected O(n + m). But a pathological input (or unlucky choice of modulus) could cause every one of the O(n) windows to hash-match the pattern, forcing an O(m) verification at each one, giving O(nm) in the worst case.</details>

**Q7.** In the N-Queens backtracking solver, what does `is_safe` check, and why is checking it *before* recursing into the next row (rather than after placing all N queens) important?

<details><summary>Show answer</summary>`is_safe` checks whether placing a queen at (row, col) would share a column or diagonal with any queen already placed in an earlier row. Checking it before recursing is what enables pruning: an unsafe partial placement is rejected immediately, so the backtracking never wastes time exploring the (often huge) space of ways to fill in the remaining rows on top of an already-invalid placement.</details>

**Q8.** What does "backtrack" mean in `columns[row] = -1` after the recursive call in `solve_n_queens`?

<details><summary>Show answer</summary>It undoes the choice made for that row (removing the queen just placed) so that the loop can try the next candidate column cleanly, as if that queen had never been placed. Without undoing the choice, a stale value from an already-explored branch would incorrectly affect later `is_safe` checks in sibling branches.</details>

**Q9.** Why is brute-force TSP described as O(n!), and why does this make it impractical for even moderately-sized instances?

<details><summary>Show answer</summary>Fixing the start city, there are (n-1)! distinct orderings of the remaining cities to try (or (n-1)!/2 if you also exploit that a tour and its reverse have equal cost), and brute force checks the cost of every one of them. Factorial growth is explosive: 10 cities is already about 181,000 tours, and 15 cities is over 43 billion, so brute force quickly becomes infeasible even though it always finds the true optimum.</details>

**Q10.** The nearest-neighbor TSP heuristic is not guaranteed to find the optimal tour. Why use it at all?

<details><summary>Show answer</summary>Because it runs in O(n^2) instead of O(n!), making it usable on problem sizes (hundreds or thousands of cities) where exact brute force is completely infeasible. In practice its tours are often reasonably close to optimal, and for many applications a fast, good-enough answer is far more useful than a guaranteed-optimal answer that would take longer than the age of the universe to compute.</details>

## MTech Extension — Deterministic worst-case-linear-time selection (median of medians)

Quickselect's O(n) running time is only an *expected* bound: a randomized pivot defeats an adversary who doesn't know your random seed, but there is no random choice that is safe against every possible input in the worst case — an unlucky sequence of pivot choices can still degrade to O(n²), just as with randomized quicksort.

The **median-of-medians** algorithm removes the randomness entirely and *guarantees* O(n) worst-case time, at the cost of a larger constant factor (so in practice, randomized quickselect is usually faster on average, and median-of-medians matters mainly when you need a *guaranteed* bound, e.g. in real-time systems). The idea:

1. Split `arr` into groups of 5.
2. Find the median of each group of 5 (cheap: sorting 5 elements is O(1) work per group).
3. Recursively find the median of those group-medians — call it the **pivot**.
4. Partition `arr` around this pivot and recurse into the appropriate side, exactly like quickselect.

The key insight that makes this O(n) rather than O(n²) in the worst case is a counting argument: because the pivot is the median of medians of groups of 5, at least roughly 3/10 of all elements are guaranteed to be less than the pivot, and at least roughly 3/10 are guaranteed to be greater — so no partition can ever be *too* lopsided, unlike plain quickselect where an adversarial input can make every partition maximally unbalanced. This bounded imbalance is what makes the recurrence solve to O(n) instead of O(n²), via the same kind of geometric-series argument used for expected-case quickselect in Section 6, except now it holds unconditionally rather than only in expectation.

In [ ]:
def median_of_five(group):
    return sorted(group)[len(group) // 2]


def median_of_medians_select(arr, k):
    """Deterministic worst-case O(n) selection of the k-th smallest element.

    Same interface as quickselect(arr, k), but the pivot is never random --
    it is guaranteed by construction to split the array reasonably evenly.
    """
    if len(arr) <= 5:
        return sorted(arr)[k]

    # Step 1-2: split into groups of 5, take each group's median.
    groups = [arr[i:i + 5] for i in range(0, len(arr), 5)]
    medians = [median_of_five(g) for g in groups]

    # Step 3: recursively find the median OF the medians -- this is the pivot,
    # and this recursive call is what makes the algorithm deterministic
    # rather than randomized (contrast with quickselect's random.choice).
    pivot = median_of_medians_select(medians, len(medians) // 2)

    # Step 4: partition around the guaranteed-good pivot and recurse, exactly
    # like quickselect's partition step.
    lows = [x for x in arr if x < pivot]
    highs = [x for x in arr if x > pivot]
    pivots = [x for x in arr if x == pivot]

    if k < len(lows):
        return median_of_medians_select(lows, k)
    elif k < len(lows) + len(pivots):
        return pivot
    else:
        return median_of_medians_select(highs, k - len(lows) - len(pivots))


# Correctness check against sorted() on both random and adversarial inputs.
random.seed(0)
for _trial in range(30):
    n = random.randint(1, 60)
    test_arr = [random.randint(-100, 100) for _ in range(n)]
    expected = sorted(test_arr)
    for k in range(n):
        assert median_of_medians_select(test_arr, k) == expected[k]

# Adversarial input: strictly increasing sequence, the kind that can break a
# poorly-chosen FIXED (non-random) pivot rule like "always pick arr[0]".
adversarial = list(range(1, 201))
for k in [0, 50, 100, 150, 199]:
    assert median_of_medians_select(adversarial, k) == adversarial[k]

print("median_of_medians_select matches sorted() on every rank, including an adversarial input")
print("Unlike quickselect, this result does not depend on any random seed at all.")

## Solutions (try the exercises yourself first!)

### Solution 1

In [ ]:
def min_parallel_fraction_for_speedup_solution(target_speedup: float) -> float:
    # 1 / (1 - p) >= target_speedup  =>  1 - p <= 1 / target_speedup  =>  p >= 1 - 1/target_speedup
    p = 1 - 1 / target_speedup
    return round(p, 4)

assert min_parallel_fraction_for_speedup_solution(2.0) == 0.5
assert min_parallel_fraction_for_speedup_solution(4.0) == 0.75
assert min_parallel_fraction_for_speedup_solution(10.0) == 0.9
print("Solution 1 verified")

### Solution 2

In [ ]:
def parallel_argmax_solution(chunks: list) -> tuple:
    partial_results = [(max(chunk), i) for i, chunk in enumerate(chunks) if chunk]
    return max(partial_results, key=lambda pair: pair[0])

assert parallel_argmax_solution([[3, 7], [10, 2], [5]]) == (10, 1)
assert parallel_argmax_solution([[9, 9, 9], [1]]) == (9, 0)
print("Solution 2 verified")

### Solution 3

In [ ]:
def classify_algorithm_solution(always_correct: bool, bounded_time: bool) -> str:
    if always_correct and not bounded_time:
        return "Las Vegas"
    if bounded_time and not always_correct:
        return "Monte Carlo"
    return "invalid"

assert classify_algorithm_solution(True, False) == "Las Vegas"
assert classify_algorithm_solution(False, True) == "Monte Carlo"
assert classify_algorithm_solution(True, True) == "invalid"
print("Solution 3 verified")

### Solution 4

In [ ]:
def estimate_pi_solution(n_samples: int) -> float:
    inside = 0
    for _ in range(n_samples):
        x = random.uniform(-1, 1)
        y = random.uniform(-1, 1)
        if x * x + y * y <= 1:
            inside += 1
    return 4 * inside / n_samples

result = estimate_pi_solution(200_000)
print(f"Solution 4 verified, estimate: {result:.4f}")

### Solution 5

In [ ]:
def top_k_smallest_solution(arr: list, k: int) -> list:
    if k <= 0:
        return []
    boundary_value = quickselect(arr, k - 1)   # the k-th smallest VALUE
    result = []
    boundary_used = False
    for x in arr:
        if x < boundary_value:
            result.append(x)
        elif x == boundary_value and not boundary_used and len(result) < k:
            # only take as many copies of the boundary value as needed to reach k
            pass
    # Simpler and equally valid: since we just need the k smallest VALUES sorted,
    # and quickselect already tells us the boundary, filter and sort directly.
    smaller_or_equal = sorted(x for x in arr if x <= boundary_value)
    # Handle duplicates of the boundary value precisely by trimming to length k
    # from a list that is guaranteed to contain at least k elements <= boundary.
    return smaller_or_equal[:k]

assert top_k_smallest_solution([7, 10, 4, 3, 20, 15], 3) == [3, 4, 7]
assert top_k_smallest_solution([9, 9, 1, 1, 5], 3) == [1, 1, 5]
assert top_k_smallest_solution([7, 2, 9, 4, 6], 0) == []
print("Solution 5 verified")

### Solution 6

In [ ]:
def count_pattern_occurrences_solution(text: str, pattern: str) -> int:
    return len(rabin_karp_search(text, pattern))

assert count_pattern_occurrences_solution("aaaa", "aa") == 3
assert count_pattern_occurrences_solution("abracadabra", "abra") == 2
assert count_pattern_occurrences_solution("abcabcabc", "abc") == 3
assert count_pattern_occurrences_solution("xyz", "abc") == 0
assert count_pattern_occurrences_solution("aaaa", "aaaa") == 1
print("✅ Solution 6 verified")

### Solution 7

In [ ]:
def count_n_queens_solutions_solution(n: int) -> int:
    count = 0
    columns = [-1] * n

    def is_safe(row, col):
        for r in range(row):
            c = columns[r]
            if c == col or abs(c - col) == abs(r - row):
                return False
        return True

    def backtrack(row):
        nonlocal count
        if row == n:
            count += 1
            return
        for col in range(n):
            if is_safe(row, col):
                columns[row] = col
                backtrack(row + 1)
                columns[row] = -1

    backtrack(0)
    return count

assert count_n_queens_solutions_solution(1) == 1
assert count_n_queens_solutions_solution(2) == 0
assert count_n_queens_solutions_solution(3) == 0
assert count_n_queens_solutions_solution(4) == 2
assert count_n_queens_solutions_solution(5) == 10
print("✅ Solution 7 verified")

### Solution 8

In [ ]:
def best_nearest_neighbor_tour_solution(distance_matrix: list) -> tuple:
    best_tour, best_cost = None, math.inf
    for start in range(len(distance_matrix)):
        tour, cost = tsp_nearest_neighbor(distance_matrix, start=start)
        if cost < best_cost:
            best_tour, best_cost = tour, cost
    return best_tour, best_cost

small_distances_check = [
    [0, 10, 15, 20, 25],
    [10, 0, 35, 25, 30],
    [15, 35, 0, 30, 20],
    [20, 25, 30, 0, 15],
    [25, 30, 20, 15, 0],
]
exact_tour_check, exact_cost_check = tsp_brute_force(small_distances_check)
best_tour_check, best_cost_check = best_nearest_neighbor_tour_solution(small_distances_check)
assert best_cost_check >= exact_cost_check
assert best_tour_check[0] == best_tour_check[-1]
assert len(set(best_tour_check[:-1])) == len(small_distances_check)
print(f"✅ Solution 8 verified (best nearest-neighbor cost: {best_cost_check}, optimal: {exact_cost_check})")